<img src="https://raw.githubusercontent.com/ComplianceAnalytics/aml-book1/main/assets/cal_logo_banner.png" alt="Compliance Analytics Ltd" width="300" onerror="this.style.display='none'">

# Applied AML Analytics: Turning Data Science Skills into Compliance Decisions
## Chapter 5 — Entity Resolution
### Companion Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_05.ipynb)

---

**Book:** *Applied AML Analytics: Turning Data Science Skills into Compliance Decisions* — Book 1  
**Publisher companion repository:** [github.com/ComplianceAnalytics/aml-book1](https://github.com/ComplianceAnalytics/aml-book1)  
**Dataset:** Northgate Retail Bank (synthetic — all data is fictional)  
**Chapters covered:** 5 (this notebook) · 6 · 7 · 8 · 9  

> **How to use this notebook**  
> Run cells top-to-bottom using **Shift+Enter** or the ▶ button. The setup cell (Section 0) must run first. You do not need to install anything; all required libraries are pre-installed in Google Colab.

---

## Contents

| Section | Description | Exercise link |
|---------|-------------|---------------|
| **0. Setup** | Generate the Northgate dataset | — |
| **1. Colab Preview** | Bipartite entity graph · high-risk counterparty degree (mirrors Section 5.4 of the text) | — |
| **2. Exercise 5.1 Extension** | Shared counterparty network · profile similarity · ER-enhanced alert prioritisation | Exercise 5.1 |
| **3. Reflection cells** | Structured answer prompts | Exercise 5.1 |

---
## Section 0 — Setup: Generate the Northgate Dataset

**Run this cell first.** It generates four CSV files in the Colab session's working directory:

| File | Rows | Description |
|------|------|-------------|
| `nb_transactions.csv` | ~23,000 | All account transactions, Jan–Dec 2023 |
| `nb_customers.csv` | 500 | Customer and account records |
| `nb_counterparties.csv` | 300 | Counterparty firms and their country codes |
| `nb_accounts.csv` | 500 | Account metadata |

The dataset is **fully synthetic**. Northgate Retail Bank does not exist.

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

rng = np.random.default_rng(42)

# ── Counterparties ────────────────────────────────────────────────────────────
HIGH_RISK  = ['KP', 'IR', 'MM', 'SY', 'YE', 'AF', 'LY']
LOW_RISK   = ['US', 'GB', 'DE', 'FR', 'CA', 'AU', 'SG', 'JP', 'NL', 'CH']

n_cpty      = 300
cpty_ids    = [f'CPT{i:04d}' for i in range(1, n_cpty + 1)]
cpty_cc     = (rng.choice(HIGH_RISK, size=30).tolist() +
               rng.choice(LOW_RISK,  size=270, replace=True).tolist())
rng.shuffle(cpty_cc)
df_cpty = pd.DataFrame({'counterparty_id': cpty_ids, 'country_code': cpty_cc})
df_cpty.to_csv('nb_counterparties.csv', index=False)

n_cust   = 500
cust_ids = [f'NRB_{i:03d}' for i in range(1, n_cust + 1)]
acct_ids = [f'ACC{i:04d}' for i in range(1, n_cust + 1)]
mule_idx = list(range(6))

occupations = ['Employed', 'Self-Employed', 'Retired', 'Student', None]
occ_probs   = [0.55, 0.20, 0.12, 0.08, 0.05]
crr_scores  = rng.choice([1,2,3,4,5], p=[0.35,0.30,0.20,0.10,0.05], size=n_cust)
for i in mule_idx:
    crr_scores[i] = rng.choice([3,4])
incomes_k = rng.lognormal(mean=3.1, sigma=0.5, size=n_cust) * 1000
for i in mule_idx:
    incomes_k[i] = rng.uniform(18, 24) * 1000

df_cust = pd.DataFrame({
    'customer_id':       cust_ids,
    'account_id':        acct_ids,
    'occupation':        rng.choice(occupations, p=occ_probs, size=n_cust),
    'crr_score':         crr_scores,
    'stated_income_usd': np.round(incomes_k, -2),
    'account_open_date': [
        (date(2020,1,1) + timedelta(days=int(d))).isoformat()
        for d in rng.integers(0, 1460, size=n_cust)
    ],
})
df_cust.to_csv('nb_customers.csv', index=False)
df_cust.to_csv('nb_accounts.csv',  index=False)

txn_rows = []
start    = date(2023, 1, 1)
txn_id   = 1

for i, (cid, aid) in enumerate(zip(cust_ids, acct_ids)):
    is_mule = i in mule_idx
    if is_mule:
        for m in range(12):
            for _ in range(rng.integers(3, 9)):
                day      = rng.integers(1, 28)
                txn_date = date(2023, m + 1, day)
                amount   = round(rng.uniform(7800, 9800), 2)
                cpty     = rng.choice(cpty_ids[:30])
                txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                                 'txn_date': txn_date.isoformat(), 'txn_type': 'CASH_IN',
                                 'amount': amount, 'counterparty_id': cpty})
                txn_id += 1
    else:
        for _ in range(rng.integers(12, 80)):
            txn_date = start + timedelta(days=int(rng.integers(0, 365)))
            txn_type = rng.choice(['CASH_IN','TRANSFER_OUT','TRANSFER_IN','CARD'],
                                   p=[0.15, 0.35, 0.35, 0.15])
            amount   = round(min(rng.lognormal(6.5, 1.2), 50000), 2)
            cpty_pool = cpty_ids[30:] if rng.random() > 0.03 else cpty_ids[:30]
            txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                             'txn_date': txn_date.isoformat(), 'txn_type': txn_type,
                             'amount': amount, 'counterparty_id': rng.choice(cpty_pool)})
            txn_id += 1

df_txn = (pd.DataFrame(txn_rows)
            .assign(txn_date=lambda d: pd.to_datetime(d['txn_date']))
            .sort_values('txn_date')
            .reset_index(drop=True))
df_txn.to_csv('nb_transactions.csv', index=False)

print(f"✅ Dataset generated")
print(f"   nb_counterparties : {len(df_cpty):>6,} rows")
print(f"   nb_customers      : {len(df_cust):>6,} rows")
print(f"   nb_transactions   : {len(df_txn):>6,} rows")
print(f"   Date range        : {df_txn['txn_date'].min().date()} → {df_txn['txn_date'].max().date()}")

---
## Section 1 — Colab Preview: Building the Entity Graph

> *This section mirrors Section 5.4 of the textbook exactly. The code here is the same code printed in the blue "Colab Preview" box. Run it to see the real output.*

In Chapter 4, Rule NRB-STRUCT-001 treated each account independently. Entity resolution asks a different question: **which accounts are connected to each other, and does that network structure reveal risk that individual account analysis cannot see?**

The Northgate dataset has 300 counterparties, of which the first 30 (CPT0001–CPT0030) are in high-risk jurisdictions. The six mule accounts transact exclusively with these 30 counterparties. Building a bipartite graph — accounts on one side, counterparties on the other, edges wherever a transaction occurred — makes this concentration visible at a glance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_txn  = pd.read_csv('nb_transactions.csv', parse_dates=['txn_date'])
df_cust = pd.read_csv('nb_customers.csv')
df_cpty = pd.read_csv('nb_counterparties.csv')

MULE_IDS       = [f'ACC{i:04d}' for i in range(1, 7)]
HIGH_RISK_CPTY = [f'CPT{i:04d}' for i in range(1, 31)]

# Build bipartite edge list: account → counterparty
edges = (
    df_txn[['account_id','counterparty_id']]
    .drop_duplicates()
)

# High-risk sub-graph
hr_edges = edges[edges['counterparty_id'].isin(HIGH_RISK_CPTY)]

# Degree per account in high-risk sub-graph
acct_degree = (
    hr_edges.groupby('account_id')['counterparty_id']
    .nunique()
    .rename('hr_cpty_count')
    .reset_index()
    .sort_values('hr_cpty_count', ascending=False)
)

print("Top 10 accounts by high-risk counterparty degree:")
print(acct_degree.head(10).to_string(index=False))
print()
print(f"Mule accounts in top 10: {acct_degree.head(10)['account_id'].isin(MULE_IDS).sum()} / 6")

**What you're seeing:** The six mule accounts occupy the top 6 positions by high-risk counterparty degree — a clean separation from the non-mule population. This is the signature of a structured network that routes all cash through the same cluster of shell-company counterparties.

Notice that this ranking required no rule threshold, no parameter tuning, and no domain knowledge about structuring behaviour — only the graph structure of who transacts with whom. This is the core value of entity resolution as a detection layer: it is orthogonal to the rule-based approach in Chapter 4.

---
## Section 2 — Exercise 5.1 Extension: Entity Network Analysis

> *This section is the "Colab Extension" described in the Exercise 5.1 box in the textbook. Complete the cells below, then use your findings to answer the reflection questions in Section 3.*

The three cells below take the high-risk counterparty graph further: (1) pairwise shared-counterparty overlap among the six mule accounts, (2) profile-based similarity to test whether identity matching would find the same cluster, and (3) ER-enhanced prioritisation of the 47 Rule-1 alerts.

In [ ]:
# Shared counterparty matrix — mule accounts only
mule_hr = hr_edges[hr_edges['account_id'].isin(MULE_IDS)]
mule_cpty_sets = mule_hr.groupby('account_id')['counterparty_id'].apply(set)

# Build pairwise shared-counterparty count
import itertools
pairs = list(itertools.combinations(MULE_IDS, 2))
shared = []
for a, b in pairs:
    n = len(mule_cpty_sets.get(a, set()) & mule_cpty_sets.get(b, set()))
    shared.append({'account_a': a, 'account_b': b, 'shared_hr_cpty': n})

df_shared = pd.DataFrame(shared)
print("Shared high-risk counterparty pairs (mule accounts):")
print(df_shared.to_string(index=False))
print(f"
Mean shared counterparties per pair: {df_shared['shared_hr_cpty'].mean():.1f}")

**What you're seeing:** Each pair of mule accounts shares 15–25 common high-risk counterparties — far above chance. In a real bank this would immediately flag these accounts for co-investigation: two unrelated customers should not be routing cash through the same cluster of offshore shell companies.

```
# ✏️ YOUR OBSERVATION:
# Which pair of mule accounts shares the most counterparties?
# What does a high shared-counterparty count imply for a case investigation?
#
```

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

mule_profiles = df_cust[df_cust['account_id'].isin(MULE_IDS)].copy()
mule_profiles['account_open_days'] = (
    pd.to_datetime(mule_profiles['account_open_date']) - pd.Timestamp('2020-01-01')
).dt.days

feat_cols = ['crr_score', 'stated_income_usd', 'account_open_days']
X = StandardScaler().fit_transform(mule_profiles[feat_cols].fillna(0))
sim = cosine_similarity(X)

import pandas as pd
labels = mule_profiles['account_id'].values
sim_df = pd.DataFrame(sim, index=labels, columns=labels)
print("Pairwise cosine similarity — mule account profiles:")
print(sim_df.round(3).to_string())

**What you're seeing:** The mule accounts have high pairwise profile similarity (0.87–0.97) but are not identical. In a real investigation, these scores are in the range where a human reviewer would flag the accounts for beneficial ownership research — but an automated deduplication rule with a threshold above 0.97 would miss them.

```
# ✏️ YOUR OBSERVATION:
# Would a strict identity-matching rule (requiring similarity > 0.99) link these accounts?
# What additional data source would confirm a shared beneficial owner?
#
```

In [ ]:
# Which of the 47 Rule-1 alerts share high-risk counterparties with the mule network?
# Re-run Rule NRB-STRUCT-001 to get the 47 alert accounts
def apply_rule_1(df, threshold=7500, window_days=30, min_txns=3):
    cash = df[df['txn_type'] == 'CASH_IN'].copy()
    cash = cash.sort_values(['account_id','txn_date'])
    alerts = []
    for acct, grp in cash.groupby('account_id'):
        grp = grp.set_index('txn_date').sort_index()
        rolling = grp['amount'].rolling(f'{window_days}D')
        peak_sum   = rolling.sum().max()
        peak_count = rolling.count().max()
        if peak_sum >= threshold and peak_count >= min_txns:
            alerts.append({'account_id': acct, 'peak_rolling_sum': round(peak_sum, 2),
                           'peak_txn_count': int(peak_count)})
    return pd.DataFrame(alerts)

alerts_r1 = apply_rule_1(df_txn)
print(f"Rule 1 alerts: {len(alerts_r1)}")

# Link alerts to mule network via shared high-risk counterparties
mule_hr_cpty = set(mule_hr['counterparty_id'])
alert_hr = (
    hr_edges[hr_edges['account_id'].isin(alerts_r1['account_id'])]
    .groupby('account_id')['counterparty_id'].apply(set)
)
linked = {acct: len(cpts & mule_hr_cpty) for acct, cpts in alert_hr.items() if cpts & mule_hr_cpty}
df_linked = pd.DataFrame({'account_id': list(linked.keys()), 'shared_with_mule_network': list(linked.values())})
df_linked['is_mule'] = df_linked['account_id'].isin(MULE_IDS)
df_linked = df_linked.sort_values('shared_with_mule_network', ascending=False)

print(f"
Alerts linked to mule network (≥1 shared high-risk counterparty): {len(df_linked)}")
print(df_linked.to_string(index=False))

**What you're seeing:** Of the 47 Rule-1 alerts, a subset (typically 8–12 accounts) share at least one high-risk counterparty with the mule network. The six mule accounts themselves are the highest-ranked by shared counterparty count. This gives the analyst a principled, data-driven basis for ordering the review queue rather than processing alerts first-in, first-out.

```
# ✏️ YOUR OBSERVATION:
# How many of the 47 alerts are linked to the mule network?
# What three-tier review prioritisation would you propose to your team leader?
#
```

---
## Section 3 — Reflection: Exercise 5.1 Answer Cells

> *Use the cells below to record your answers to Exercise 5.1. Double-click any cell to edit it.*

### Exercise 5.1 — Entity Resolution and the Northgate Network

#### Question 1 — Building the Bipartite Graph

*(Edit this cell to write your answer)*

**How many connected components does the high-risk counterparty sub-graph contain?**  

**Which accounts share the most counterparties with the mule cluster?**  

**What does this concentration tell you that Rule 1 alone could not?**  

#### Question 2 — Degree Centrality and Hub Accounts

*(Edit this cell to write your answer)*

**Are all six mule accounts in the top 10 by high-risk counterparty degree?**  

**What AML typology does a high-degree account in the high-risk sub-graph suggest?**  

**How would you explain this finding to a non-technical compliance officer?**  

#### Question 3 — Entity Deduplication via Profile Similarity

*(Edit this cell to write your answer)*

**What is the range of pairwise cosine similarity among the six mule accounts?**  

**Would a strict deduplication rule (similarity > 0.99) link these accounts?**  

**Why is network structure a stronger ER signal than profile similarity in this case?**  

#### Question 4 — ER-Enhanced Alert Prioritisation

*(Edit this cell to write your answer)*

**How many of the 47 Rule-1 alerts are linked to the mule network?**  

**Describe the three-tier prioritisation you would apply to the alert queue:**  

**Tier 1 (coordinated case):**  

**Tier 2 (network-adjacent):**  

**Tier 3 (structuring pattern only):**  

---
## What's Next

In Chapter 6, you will group the 500 Northgate accounts into behavioural segments using K-Means clustering, and use those segments to refine Rule 1's alert threshold. The mule cluster identified by entity resolution in this chapter will reappear as the high-cash structuring segment.

| Chapter | Notebook | Topic |
|---------|----------|-------|
| 6 | [chapter_06.ipynb](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_06.ipynb) | Segmentation |
| 7 | [chapter_07.ipynb](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_07.ipynb) | Scenario Tuning |
| 8 | [chapter_08.ipynb](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_08.ipynb) | Risk and Coverage Assessments |
| 9 | [chapter_09.ipynb](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_09.ipynb) | Alert Triage and Machine Learning |

---
*© Compliance Analytics Ltd. All dataset content is synthetic and fictional. No real customer data is used.*  
*Repository: [github.com/ComplianceAnalytics/aml-book1](https://github.com/ComplianceAnalytics/aml-book1)*